# Phase 3: Model Development & Hyperparameter Tuning
## IT24102403 — Uthmisha M.H.I

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import classification_report
import joblib
from xgboost import XGBClassifier
import os

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
# Load preprocessed data (imbalanced for pipeline)
X_train = pd.read_csv('preprocessed/X_train_imbalanced.csv')
y_train = pd.read_csv('preprocessed/y_train_imbalanced.csv').squeeze()
X_test = pd.read_csv('preprocessed/X_test.csv')
y_test = pd.read_csv('preprocessed/y_test.csv').squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}\n")

print("Class distribution of y_train:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution of y_test:")
print(y_test.value_counts(normalize=True))

X_train shape: (36168, 42)
y_train shape: (36168,)
X_test shape: (9043, 42)
y_test shape: (9043,)

Class distribution of y_train:
y
0    0.883018
1    0.116982
Name: proportion, dtype: float64

Class distribution of y_test:
y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


## 3.1 Cross-Validation Strategy

In [3]:
# Define stratified cross-validation strategy
# StratifiedKFold maintains class proportions in each fold
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3.2 Baseline Model — Logistic Regression

In [4]:
# Create Logistic Regression baseline using SMOTE pipeline
lr_baseline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit and predict
lr_baseline.fit(X_train, y_train)
y_pred_lr = lr_baseline.predict(X_test)
y_pred_proba_lr = lr_baseline.predict_proba(X_test)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.84      0.88      7985
           1       0.27      0.46      0.34      1058

    accuracy                           0.79      9043
   macro avg       0.60      0.65      0.61      9043
weighted avg       0.85      0.79      0.81      9043



In [5]:
# Hyperparameter tuning for Logistic Regression
lr_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

lr_param_grid = {
    'lr__C': [0.01, 0.1, 1, 10],
    'lr__penalty': ['l1', 'l2'],
    'lr__solver': ['liblinear']
}

lr_grid = GridSearchCV(lr_pipeline,
                       param_grid=lr_param_grid,
                       scoring='f1',
                       cv=cv_strategy,
                       n_jobs=-1)

lr_grid.fit(X_train, y_train)
print(f"Best parameters for Logistic Regression: {lr_grid.best_params_}")
print(f"Best CV F1-score: {lr_grid.best_score_:.4f}")

Best parameters for Logistic Regression: {'lr__C': 0.01, 'lr__penalty': 'l1', 'lr__solver': 'liblinear'}
Best CV F1-score: 0.3697


## 3.3 Decision Tree Classifier

In [6]:
# Create Decision Tree baseline using SMOTE pipeline
dt_baseline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42))
])

# Fit and predict
dt_baseline.fit(X_train, y_train)
y_pred_dt = dt_baseline.predict(X_test)

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.86      0.89      7985
           1       0.25      0.35      0.30      1058

    accuracy                           0.80      9043
   macro avg       0.58      0.61      0.59      9043
weighted avg       0.83      0.80      0.82      9043



In [8]:
# Hyperparameter tuning for Decision Tree
dt_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    'dt__max_depth': [3, 5, 7, 10, 15, None],
    'dt__min_samples_split': [2, 5, 10, 20],
    'dt__min_samples_leaf': [1, 2, 5, 10]
}

dt_grid = GridSearchCV(dt_pipeline,
                       param_grid=dt_param_grid,
                       scoring='f1',
                       cv=cv_strategy,
                       n_jobs=-1)

dt_grid.fit(X_train, y_train)
print(f"Best parameters for Decision Tree: {dt_grid.best_params_}")
print(f"Best CV F1-score: {dt_grid.best_score_:.4f}")

Best parameters for Decision Tree: {'dt__max_depth': 7, 'dt__min_samples_leaf': 1, 'dt__min_samples_split': 2}
Best CV F1-score: 0.3983


## 3.4 Random Forest Classifier

In [9]:
# Random Forest baseline using SMOTE pipeline
rf_baseline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit and predict
rf_baseline.fit(X_train, y_train)
y_pred_rf = rf_baseline.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.94      0.93      7985
           1       0.45      0.38      0.41      1058

    accuracy                           0.87      9043
   macro avg       0.69      0.66      0.67      9043
weighted avg       0.86      0.87      0.87      9043



In [10]:
# Hyperparameter tuning for Random Forest using RandomizedSearchCV
rf_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
])

rf_param_dist = {
    'rf__n_estimators': [50, 100, 200, 300],
    'rf__max_depth': [5, 10, 15, 20, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5]
}

rf_random = RandomizedSearchCV(rf_pipeline,
                               param_distributions=rf_param_dist,
                               n_iter=30,
                               scoring='f1',
                               cv=cv_strategy,
                               random_state=42,
                               n_jobs=-1)

rf_random.fit(X_train, y_train)
print(f"Best parameters for Random Forest: {rf_random.best_params_}")
print(f"Best CV F1-score: {rf_random.best_score_:.4f}")

Best parameters for Random Forest: {'rf__n_estimators': 300, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_depth': 20}
Best CV F1-score: 0.4411


## 3.5 XGBoost Classifier

In [11]:
# XGBoost baseline using SMOTE pipeline
xgb_baseline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# Fit and predict
xgb_baseline.fit(X_train, y_train)
y_pred_xgb = xgb_baseline.predict(X_test)

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      7985
           1       0.53      0.32      0.40      1058

    accuracy                           0.89      9043
   macro avg       0.72      0.64      0.67      9043
weighted avg       0.87      0.89      0.87      9043



In [12]:
# Hyperparameter tuning for XGBoost using RandomizedSearchCV
xgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# Note the 'xgb__' prefix added to the parameters
xgb_param_dist = {
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgb__max_depth': [3, 5, 7, 10],
    'xgb__n_estimators': [100, 200, 300],
    'xgb__subsample': [0.7, 0.8, 0.9, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

xgb_random = RandomizedSearchCV(xgb_pipeline,
                                param_distributions=xgb_param_dist,
                                n_iter=30,
                                scoring='f1',
                                cv=cv_strategy,
                                random_state=42,
                                n_jobs=-1)

xgb_random.fit(X_train, y_train)
print(f"Best parameters for XGBoost: {xgb_random.best_params_}")
print(f"Best CV F1-score: {xgb_random.best_score_:.4f}")

Best parameters for XGBoost: {'xgb__subsample': 1.0, 'xgb__n_estimators': 300, 'xgb__max_depth': 10, 'xgb__learning_rate': 0.01, 'xgb__colsample_bytree': 0.9}
Best CV F1-score: 0.4423


## 3.6 Retrain All Models with Best Hyperparameters

In [13]:
# Retrain all models with best hyperparameters
best_lr = lr_grid.best_estimator_
best_dt = dt_grid.best_estimator_
best_rf = rf_random.best_estimator_
best_xgb = xgb_random.best_estimator_

models = {
    'Logistic Regression': best_lr,
    'Decision Tree': best_dt,
    'Random Forest': best_rf,
    'XGBoost': best_xgb
}

results_dict = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    results_dict[name] = {'y_pred': preds, 'y_pred_proba': probs}

## 3.7 Cross-Validation Comparison

In [14]:
# Cross-Validation Comparison
cv_results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv_strategy, scoring='f1', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'Mean F1': np.mean(scores),
        'Std F1': np.std(scores)
    })

cv_comparison_df = pd.DataFrame(cv_results)
print(cv_comparison_df)

                 Model   Mean F1    Std F1
0  Logistic Regression  0.369686  0.003208
1        Decision Tree  0.398295  0.015948
2        Random Forest  0.441108  0.010420
3              XGBoost  0.442253  0.006179


## 3.8 Save Models and Predictions

In [15]:
os.makedirs('models', exist_ok=True)
os.makedirs('predictions', exist_ok=True)
# Save models
for name, model in models.items():
    file_name = name.replace(' ', '_').lower()
    joblib.dump(model, f'models/{file_name}_model.pkl', compress=3)
# Save predictions
all_results = []
for name, res in results_dict.items():
    # create DataFrame with predictions
    all_results.append({
        'Model': name,
        'y_pred': res['y_pred'].tolist(),
        'y_pred_proba': res['y_pred_proba'].tolist()
    })
results_df = pd.DataFrame(all_results)
results_df.to_csv('predictions/all_results.csv', index=False)
print("Models and predictions saved successfully.")


Models and predictions saved successfully.


## 3.9 Model Development Summary
Four classification models (Logistic Regression, Decision Tree, Random Forest, XGBoost) were successfully trained using an imbalanced-learn pipeline that dynamically applied SMOTE during cross-validation. For each model, baseline performance was evaluated and hyperparameter tuning was conducted to optimize the F1-score via 5-fold stratified cross-validation (using `GridSearchCV` or `RandomizedSearchCV`). The retrained models using the optimal configurations were cross-validated to compare robustness. Finally, the best models were saved, and their predictions (class labels and probabilities) on the test set were exported for detailed formal evaluation in the subsequent phase.